## 各船の船舶ログから偏流を計算するまでの流れ
1. 複数のログファイルをFileConection.ipynbで１つのファイルにする
1. S1-ShipLogToS1でファイルから必要な情報を抜き出す
1. このファイル(環境省の船舶ログから偏流)で各船の偏流を計算

## import files

In [132]:
%matplotlib inline
import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from filterpy.gh import GHFilter
from numpy.random import randn
import seaborn as sns
from tqdm import tqdm
import pickle as pkl
import math
import os 
import os.path as osp
import re

import warnings
warnings.simplefilter('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid", palette="muted", color_codes=True)

from utils import *
from kf_params import *

## 関数の定義

In [1]:
def latlon_to_mesh_df(lat, lon, deg_per_mesh=1/36, size=[1050, 1191], latlon_range=[20-1/36, 117-1/36]):
    # lat, lon -> [lon, lat]
    grid0 = ((lon-latlon_range[1]).astype(int)/deg_per_mesh)
    grid1 = ((lat-latlon_range[0]).astype(int)/deg_per_mesh)
    grid0[grid0 < 0] = -1
    grid0[grid0 > size[0]] = -1
    grid1[grid1 < 0] = -1
    grid1[grid1 > size[1]] = -1
    return grid0, grid1

def mean_ground_speed(dt, lat1, lon1, lat2, lon2):
    
    lat1 = dms_to_deg(lat1)
    lon1 = dms_to_deg(lon1)
    
    lat2 = dms_to_deg(lat2)
    lon2 = dms_to_deg(lon2)
    
    dist, deg = dist_deg_latlon(lat1, lon1, lat2, lon2)
    speed = dist/dt
    return ((lat1+lat2)/2, (lon1+lon2)/2), (speed, deg)

def dist_deg_latlon(lat1, lon1, lat2, lon2):
    from geopy.distance import geodesic
    res = geodesic((lat1, lon1), (lat2, lon2))
    
    # 方位角を計算
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    delta_lon = lon2_rad - lon1_rad
    y = math.sin(delta_lon) * math.cos(lat2_rad)
    x = math.cos(lat1_rad) * math.sin(lat2_rad) - math.sin(lat1_rad) * math.cos(lat2_rad) * math.cos(delta_lon)
    bearing = math.degrees(math.atan2(y, x))

    # 0から360度の範囲に調整
    bearing = (bearing + 360) % 360
    return res.meters, bearing

def dms_to_deg(x):
    deg = x // 100
    mit = x - deg*100
    #print(x)
    #print(deg)
    #print(mit)
    #print(sec)
    return deg + mit/60

def deg_to_rat(deg):
    #  北基準のdef
    return (-deg+90)*np.pi/180

def remove_outlier(df, key):
    # 下位・上位５％のデータを消す（外れ値対策）
    q1 = df[key].quantile(0.05)
    q2 = df[key].quantile(0.95)
    tf = (df[key]>q1) & (df[key]<q2)
    df2 = df[tf]
    return df2
    
def latlon_knot(lat1, lon1, lat2, lon2):
    distance = haversine_distance(lat1, lon1, lat2, lon2)/60
    knot = (distance/(time_set[i+1]-time_set[i])) * 1.94384
    return knot

In [134]:
path = r"E:\shunsukeE\data\shiplog/"
files = os.listdir(path)
dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]

In [135]:
year = dt_year = 2015
month = dt_month = 9
dt_day = 1
n_day = 1 #nday_month(dt_month)
target_ship = '87東洋'

In [146]:
for i in range(1,1): 
    print(i)

In [147]:
path = r"E:\shunsukeE\data\shiplog/"
files = os.listdir(path)
dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]

patterns = []
for day in range(1, n_day+1):
    patterns.append(fr'(\w+){month:02}{day:02}.slog1')
forbid_patterns = [fr'(\w+).slog1err']

log_datas = []
path_name = []
path_logs = []
path2 = osp.join(path, target_ship, '2015')
try:
    files = os.listdir(path2)
except:
    print(f'not found {path2}')
filenames = [f for f in files if os.path.isfile(osp.join(path2, f))]
for pattern in patterns:
    for f in filenames:
        if re.match(pattern, f):
            forbid = False
            for forbid_pattern in forbid_patterns:
                if re.match(forbid_pattern, f):
                   forbid = True
                   break
            if not forbid:
                f_path = osp.join(path2, f)
                try:
                    log = pd.read_csv(f_path, encoding="cp932", header=None)
                    log.columns = ["UTC", "NMEA", "data1", "data2"]
                    log_datas.append(log)
                    path_name.append(f_path)
                    path_logs.append(path2)
                    print(f_path)
                except:
                    print(f'Error {f_path}')

        else:
            continue


E:\shunsukeE\data\shiplog/87東洋\2015\第八十七東洋丸FileOut20150901.slog1


In [152]:
path = r"E:\shunsukeE\data\shiplog/"
files = os.listdir(path)
dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]

patterns = []
for day in range(1, n_day+1):
    patterns.append(fr'(\w+){month:02}{day:02}.slog1')
forbid_patterns = [fr'(\w+).slog1err']

log_datas = []
path_name = []
path_logs = []
for target_ship in dirs:
    path2 = osp.join(path, target_ship, '2015')
    try:
        files = os.listdir(path2)
    except:
        print(f'not found {path2}')
    filenames = [f for f in files if os.path.isfile(osp.join(path2, f))]
    for pattern in patterns:
        for f in filenames:
            print(f)
            path_logs.append(f)
            break

中春丸FileOut20150101-1130.txt
第二辰巳丸FileOut20151001-1020.txt
第十一和光丸FileOut20150103-0620.txt
第十八英山丸FileOut20150103-1420.txt
第二十一東丸FileOut20150729-1410.txt
第三十三東洋丸FileOut20150102-0340.txt
第八十七東洋丸FileOut20150106-1820.txt
SUNNYMARSFileOut20150104-0430.txt
ひまわり２FileOut20150118-0730.txt
興春丸FileOut20150109-1130.txt
黒潮丸FileOut20150103-1340.txt
昇山丸FileOut20150909-1850.txt
昭建丸FileOut20150105-0230.txt
昭瑞丸FileOut20150929-1640.txt
清栄丸FileOut20150102-1130.txt
双信丸FileOut20150109-0740.txt
筑前丸FileOut20150101-2100.txt
如月丸FileOut20150118-0650.txt
第八菱洋丸FileOut20150123-0740.txt
豊鶴丸FileOut20150105-0400.txt
立眞丸FileOut20150102-1140.txt


In [137]:
print(f'loaded file num: {len(log_datas)}')

loaded file num: 0


In [138]:
for i in range(len(log_datas)):
    log_data = log_datas[i]
    print(f'--------num data--------')
    print(f"FileName: {path_name[i]}")
    print(f'GGA: {len(log_data[log_data["NMEA"]=="GGA"])}')
    print(f'VBW: {len(log_data[log_data["NMEA"]=="VBW"])}')
    print(f'HDT: {len(log_data[log_data["NMEA"]=="HDT"])}')
    print(f'VTG: {len(log_data[log_data["NMEA"]=="VTG"])}')
    print(f'VHW: {len(log_data[log_data["NMEA"]=="VHW"])}')
    print(f'------------------------\n')

In [117]:
def divide_nmea(log_data):
    # 時間の整理　dtIdx: 0時からの経過時間，dtIdx_Minute:0時0分からの経過分
    time_utc =log_data["UTC"].values
    str_format = '%Y/%m/%d %H:%M:%S'
    epoc_dt = datetime.datetime(dt_year, dt_month, dt_day, 0, 0, 0)
    time_idx = []
    time_idx2 = []
    tf = []
    for t in time_utc:
        idx = datetime.datetime.strptime(t, str_format) 
        idx2 = idx - epoc_dt
        time_idx.append(int(idx2.days*24 + idx2.seconds / (60*60)))
        time_idx2.append(int(idx2.days*24*60 + idx2.seconds / (60)))
    time_idx = np.array(time_idx)
    time_idx2 = np.array(time_idx2)
    log_data["DtIdx"] = time_idx
    log_data["DtIdx_Minute"] = time_idx2

    # NMEAデータの抽出
    # ggaデータの抽出
    gga = log_data[log_data["NMEA"]=="GGA"]
    gga.columns = ['UTC', 'NMEA', 'LatDMS', 'LonDMS', 'DtIdx', 'DtIdx_Minute']
    gga['Lat'] = dms_to_deg(gga['LatDMS'])
    gga['Lon'] = dms_to_deg(gga['LonDMS'])

    # vbwデータの抽出
    vbw = log_data[log_data["NMEA"]=="VBW"]
    vbw.columns = ['UTC', 'NMEA', 'LonWaterSpeed', 'TraWaterSpeed', 'DtIdx', 'DtIdx_Minute']

    
    # hdtデータの抽出
    hdt = log_data[log_data["NMEA"]=="HDT"]
    hdt = hdt.drop('data2', axis=1)
    hdt.columns = ['UTC', 'NMEA', 'HeadDeg', 'DtIdx', 'DtIdx_Minute']

    # vtgデータの抽出    
    vtg = log_data[log_data["NMEA"]=="VTG"]
    vtg.columns = ['UTC', 'NMEA', 'HeadDeg', 'GroundSpeed', 'DtIdx', 'DtIdx_Minute']

    # vtgデータの抽出
    vhw = log_data[log_data["NMEA"]=="VHW"]
    vhw.columns = ['UTC', 'NMEA', 'HeadDeg', 'WaterSpeed', 'DtIdx', 'DtIdx_Minute']
    
    # データ処理
    delete_time_nmri = {}
    stop_knot = 0.5
    
    # vtgデータ処理
    prev_dt = set(vtg['DtIdx_Minute'].values)
    tf = vtg['GroundSpeed']>stop_knot
    vtg = vtg[tf]
    delete_time_nmri['VTG'] = prev_dt - set(vtg['DtIdx_Minute'].values)

    # vbwのデータ処理
    prev_dt = set(vbw['DtIdx_Minute'].values)
    tf = vbw['LonWaterSpeed']!=-999.0
    vbw = vbw[tf] 
    tf = vbw['LonWaterSpeed']>stop_knot
    vbw = vbw[tf] 
    delete_time_nmri['VBW'] = prev_dt - set(vbw['DtIdx_Minute'].values)
    
    # hdtデータ処理
    prev_dt = set(hdt['DtIdx_Minute'].values)
    time_set = sorted(set(hdt["DtIdx_Minute"]))
    delete_range = 10
    thres = (10/180)*np.pi
    for i in range(len(time_set)-1):
            hdt1 = hdt[time_set[i] == hdt["DtIdx_Minute"]]
            hdt2 = hdt[time_set[i+1] == hdt["DtIdx_Minute"]]
            
            hdt1 = remove_outlier(hdt1, 'HeadDeg')
            hdt2 = remove_outlier(hdt2, 'HeadDeg')
            
            headRat1 = deg_to_rat(hdt1['HeadDeg'])
            sin1 = np.mean(np.sin(headRat1))
            cos1 = np.mean(np.cos(headRat1))
            
            headRat2 = deg_to_rat(hdt2['HeadDeg'])
            sin2 = np.mean(np.sin(headRat2))
            cos2 = np.mean(np.cos(headRat2))

            delta_theta = np.arccos(sin1*sin2 + cos1*cos2)
            omega = delta_theta/(time_set[i+1]-time_set[i])
            
            time = time_set[i]
            if np.abs(omega)>thres:
                tf = hdt['DtIdx_Minute']!=time
                hdt = hdt[tf]
                for j in range(1, 10):
                    tf = hdt['DtIdx_Minute']!=time-j
                    hdt = hdt[tf]
                    tf = hdt['DtIdx_Minute']!=time+j
                    hdt = hdt[tf]
    delete_time_nmri['HDT'] = prev_dt - set(hdt['DtIdx_Minute'].values)

    # vhwデータ処理
    prev_dt = set(vhw['DtIdx_Minute'].values)
    tf = vhw['WaterSpeed']!=-999.0
    vhw = vhw[tf]
    tf = vhw['WaterSpeed']>stop_knot
    vhw = vhw[tf]
    delete_time_nmri['VHW'] = prev_dt - set(vhw['DtIdx_Minute'].values)
    
    return gga, vbw, hdt, vtg, vhw, delete_time_nmri


In [118]:
ggas = []
vbws = []
hdts = []
vtgs = []
vhws = []
delete_times = []

for i in range(len(log_datas)):
    gga, vbw, hdt, vtg, vhw,  delete_time_nmri = divide_nmea(log_datas[i])
    ggas.append(gga)
    vbws.append(vbw)
    hdts.append(hdt)
    vtgs.append(vtg)
    vhws.append(vhw)
    delete_times.append(delete_time_nmri)

In [119]:
# for i in range(len(delete_times)):
#     delete_times[i] = set(delete_times[i])

## データの確認

In [120]:
#gga, vbw, hdt, vtg, delete_time_gga, delete_time_vbw, delete_time_hdt = divide_nmea(log_datas[0])

In [121]:
for i in range(len(log_datas)):
    log_data = log_datas[i]
    print(f'--------num data--------')
    print(f"FileName: {path_name[i]}")
    print(f'GGA: {len(ggas[i])}')
    print(f'VBW: {len(vbws[i])}')
    print(f'HDT: {len(hdts[i])}')
    print(f'VTG: {len(vtgs[i])}')
    print(f'VHW: {len(vhws[i])}')
    print(f'------------------------\n')

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150901.slog1
GGA: 90442
VBW: 52257
HDT: 69570
VTG: 105771
VHW: 0
------------------------

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150902.slog1
GGA: 85163
VBW: 76191
HDT: 80885
VTG: 152524
VHW: 0
------------------------

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150903.slog1
GGA: 84096
VBW: 69980
HDT: 70343
VTG: 107556
VHW: 0
------------------------

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150904.slog1
GGA: 85632
VBW: 50631
HDT: 72009
VTG: 70409
VHW: 0
------------------------

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150905.slog1
GGA: 92612
VBW: 79187
HDT: 82716
VTG: 160358
VHW: 0
------------------------

--------num data--------
FileName: E:\shunsukeE\data\shiplog/33東洋\2015\第三十三東洋丸FileOut20150906.slog1
GGA: 59418


In [122]:
show=False
if show:
    plt.rcParams['font.family'] = 'MS Gothic' 
    fig, axes = plt.subplots(nrows=len(ggas), ncols=4, figsize=(32, 8*len(ggas)))

    for i in range(len(ggas)):
        print('--------------------')
        print(f'path: {path_name[i]}')
        axes[i, 0].set_title(path_name[i])
        for axi in range(3):
            ax = axes[i, axi]
            ggas[i].plot.scatter(x='Lon', y='Lat',
                            marker='s', c='r', s=50, alpha=0.5, ax=ax, label='Deleted data')

        ax = axes[i, 0]
        time_set = set(ggas[i]["DtIdx_Minute"].values)
        gga1 = ggas[i]
        for time in time_set:
            if not time in vtgs[i]["DtIdx_Minute"].values:
                tf = gga1['DtIdx_Minute']!=time
                gga1 = gga1[tf]
        gga1.plot.scatter(x='Lon', y='Lat',
            marker='s', c='b', s=50, alpha=0.5, ax=ax, label='available VTG')
        print(f'VTG: {len(gga1)}')

        ax = axes[i, 1]
        time_set = set(ggas[i]["DtIdx_Minute"].values)
        gga1 = ggas[i]
        for time in time_set:
            if not time in hdts[i]["DtIdx_Minute"].values:
                tf = gga1['DtIdx_Minute']!=time
                gga1 = gga1[tf]
        gga1.plot.scatter(x='Lon', y='Lat',
            marker='s', c='b', s=50, alpha=0.5, ax=ax, label='available HDT')
        print(f'HDT: {len(gga1)}')

        ax = axes[i, 2]
        time_set = set(ggas[i]["DtIdx_Minute"].values)
        gga1 = ggas[i]
        for time in time_set:
            if not time in vbws[i]["DtIdx_Minute"].values and not time in vhws[i]["DtIdx_Minute"].values:
                tf = gga1['DtIdx_Minute']!=time
                gga1 = gga1[tf]
        gga1.plot.scatter(x='Lon', y='Lat',
            marker='s', c='b', s=50, alpha=0.5, ax=ax, label='available VBW or VHW')
        print(f'VBW, VHW: {len(gga1)}')

        ax = axes[i, 3]
        time_set = set(ggas[i]["DtIdx_Minute"].values)
        gga1 = ggas[i]
        limx = (np.max(gga1['Lon'])-np.min(gga1['Lon']))*0.1
        ax.set_xlim(np.min(gga1['Lon'])-limx, np.max(gga1['Lon'])+limx)
        limy = (np.max(gga1['Lat'])-np.min(gga1['Lat']))*0.1
        ax.set_ylim(np.min(gga1['Lat'])-limy, np.max(gga1['Lat'])+limy)
        for time in time_set:
            if (not time in set(vbws[i]["DtIdx_Minute"].values) and not time in set(vhws[i]['DtIdx_Minute'].values)) or \
                not time in set(hdts[i]["DtIdx_Minute"].values) or not time in set(vtgs[i]["DtIdx_Minute"].values):
                tf = gga1['DtIdx']!=time
                gga1 = gga1[tf]
        gga1.plot.scatter(x='Lon', y='Lat',
            marker='s', c='b', s=50, alpha=0.5, ax=ax, label='exist data (gga, vbw, hdt, vtg)')
        print(f'ALL: {len(gga1)}')
    plt.show()

## 偏流の計算

In [123]:
def cur_nmea(gga, vbw, hdt, vtg):
    min_timeHours = np.min(gga["DtIdx"])
    max_timeHours = np.max(gga["DtIdx"])
    num_timeHours = max_timeHours - min_timeHours + 1

    grids0 = []
    grids1 = []
    curN = []
    curE = []
    timeMinutes = []
    timeHours = []
    UTC_time = []
    lats = []
    lons = []
    curN_grid = {}
    curE_grid = {}
    count_grid = {}

    time_set_h = set(gga["DtIdx"])
    for s in time_set_h:
        curN_grid[s] = np.zeros(nan_map.shape)
        curE_grid[s] = np.zeros(nan_map.shape)
        count_grid[s] = np.zeros(nan_map.shape) 

    time_set = set(gga["DtIdx_Minute"])
    for time in time_set:
        print('--------------------')
        gga1 = gga[time == gga["DtIdx_Minute"]]
        vbw1 = vbw[time == vbw["DtIdx_Minute"]]
        hdt1 = hdt[time == hdt["DtIdx_Minute"]]
        vtg1 = vtg[time == vtg["DtIdx_Minute"]]
        if (len(vbw1)==0 or len(gga1)==0 or len(hdt1)==0 or len(vtg1)==0): 
            if len(vbw1)==0:
                print('nodata vbw')
            if len(gga1)==0:
                print('nodata gga') 
            if len(hdt1)==0:
                print('nodata hdt') 
            if len(vtg1)==0:
                print('nodata vtg') 
            continue

        # GGAからLat Lon 取得
        lat1 = np.mean(remove_outlier(gga1, 'Lat')['Lat'])
        lon1 = np.mean(remove_outlier(gga1, 'Lon')['Lon'])
        # データ数が少なすぎると、外れ値消去の際にnanになる，データが少ないときはその時間は計算せずcontinue
        if not lat1==lat1 or not lon1==lon1:
            print('not enought gga data')
            continue

        # HDTから船首方位取得
        headRat = deg_to_rat(hdt1['HeadDeg'])
        sin1 = np.mean(np.sin(headRat))
        cos1 = np.mean(np.cos(headRat))
        

        # VBWから対船水速取得
        water_speed = np.mean(remove_outlier(vbw1, 'LonWaterSpeed')['LonWaterSpeed'])
        if not water_speed==water_speed:
            print('not enought vbw data')
            continue

        # VTGから船首方位・対地船速取得
        g_headRat = deg_to_rat(vtg1['HeadDeg'])
        g_sin1 = np.mean(np.sin(g_headRat))
        g_cos1 = np.mean(np.cos(g_headRat))
        ground_speed = np.mean(remove_outlier(vtg1, 'GroundSpeed')['GroundSpeed'])
        if not ground_speed==ground_speed:
            print('not enought vtg data')
            continue

        # 偏流の計算
        curN.append(ground_speed*g_sin1 - water_speed*sin1)
        curE.append(ground_speed*g_cos1 - water_speed*cos1)
        #curN.append(-water_speed*sin1 + ground_speed*g_sin1)
        #curE.append(-water_speed*cos1 + ground_speed*g_cos1)
        print(f'N {ground_speed*g_sin1 - water_speed*sin1}')
        print(f'E {ground_speed*g_cos1 - water_speed*cos1}')
        
        # Gridの位置計算
        grid0, grid1 = latlon_to_mesh(lat1, lon1)

        grids0.append(grid0)
        grids1.append(grid1)
        lats.append(lat1)
        lons.append(lon1)

        UTC_time.append(gga1['UTC'].values[-1][:-6])
        timeMinutes.append(time)
        timeHours.append(gga1['DtIdx'].values[-1])
        curN_grid[timeHours[-1]][grid0][grid1] += curN[-1]
        curE_grid[timeHours[-1]][grid0][grid1] += curE[-1]
        count_grid[timeHours[-1]][grid0][grid1] += 1
        print('--------------------')

    grid_cur_m = pd.DataFrame([])
    grid_cur_m["DtIdx"] = timeHours
    grid_cur_m["DtIdx_Minute"] = timeMinutes
    grid_cur_m["UTC"] = UTC_time
    grid_cur_m["CurN"] = curN
    grid_cur_m["CurE"] = curE
    grid_cur_m["Grid0"] = grids0
    grid_cur_m["Grid1"] = grids1
    grid_cur_m["Lat"] = lats
    grid_cur_m["Lon"] = lons

    for i in time_set_h:
        count_grid[i][count_grid[i] == 0] += 1
        curN_grid[i] = curN_grid[i]/count_grid[i]
        curE_grid[i] = curE_grid[i]/count_grid[i]
    return grid_cur_m, curN_grid, curE_grid


In [124]:
grid_cur_minute = []
curN_grid = []
curE_grid = []
for i in range(len(ggas)):
    grid_cur, N, E = cur_nmea(ggas[i], vbws[i], hdts[i], vtgs[i])
    grid_cur_minute.append(grid_cur)
    curN_grid.append(N)
    curE_grid.append(E)

--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
nodata vtg
--------------------
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata hdt
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
noda

nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw


N 0.9003642749272558
E -0.4351304773908593
--------------------
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
not enought vtg data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought v

not enought vbw data
--------------------
N -0.083904036773923
E 0.5502527852250712
--------------------
--------------------
N -0.07471554567798222
E 0.576787499500945
--------------------
--------------------
N -0.12391798510772745
E 0.5810279294582141
--------------------
--------------------
not enought vbw data
--------------------
N -0.13103564436847925
E 0.7921300635146586
--------------------
--------------------
N -0.19625361077916814
E 0.8273128122056423
--------------------
--------------------
N -0.34027752428437985
E 0.7682791053236553
--------------------
--------------------
N -0.41300478114274064
E 0.8257286859321002
--------------------
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
N -0.4048378165995494
E 0.6905658548408606
--------------------
--------------------
N -0.4325738224061908
E 0.7050527570543323
--------------------
--------------------
N -0.4209518770027376

N -0.06429865279874925
E -0.33908054916682673
--------------------
--------------------
N -0.06396614889856167
E -0.33974973521288554
--------------------
--------------------
N -0.11317512805021579
E -0.2975019498685558
--------------------
--------------------
N -0.0883817110835956
E -0.2725955143017167
--------------------
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
N 0.021757197350947166
E -0.27903391196870864
--------------------
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
N 0.029280949383789334
E -0.23361409866799754
--------------------
--------------------
not enought vbw data
--------------------
N -0.0625135334112219
E -0.20882955855464402
--------------------
--------------------
not enought vtg data
--------------------
not enought vbw data
--------------------
not enough

...

N -0.6950478308845742
E 1.032413480533899
--------------------
--------------------
N -0.5949881444345895
E 1.0143445441510295
--------------------
--------------------
N -0.6963410412891609
E 1.2350918679681762
--------------------
--------------------
N -0.7066406108464047
E 1.2359493697227935
--------------------
--------------------
N -0.842942240971821
E 1.054380676948858
--------------------
--------------------
N -0.8299035813946798
E 0.7071225909296421
--------------------
--------------------
N -0.7883834596943906
E 0.5607528084975089
--------------------
--------------------
N -0.9131759563208206
E 0.6219346406952013
--------------------
--------------------
N -0.9756778042105481
E 0.6147520267419502
--------------------
--------------------
N -0.7791606695935496
E 0.5185413752923429
--------------------
--------------------
N -0.7890134103165067
E 0.4439062136040235
--------------------
--------------------
N -0.7345030796509757
E 0.4972936371257841
--------------------
----

N 2.0721169541006272
E 0.45681176408347923
--------------------
--------------------
N 2.134434233906756
E 0.5415281506327965
--------------------
--------------------
N 2.2163120423559075
E 0.45334155431774104
--------------------
--------------------
N 2.294759736985119
E 0.5381589128533202
--------------------
--------------------
N 2.191468434681493
E 0.5256874369004549
--------------------
--------------------
N 2.2586025434384567
E 0.5532830441122503
--------------------
--------------------
N 2.2809650414377423
E 0.13290217192897913
--------------------
--------------------
N 2.4163159590213983
E 0.06399671029353105
--------------------
--------------------
not enought vbw data
--------------------
not enought vbw data
--------------------
N 2.6024191537559798
E -0.17032515296737483
--------------------
--------------------
N 2.5862827530541175
E -0.3347853682893165
--------------------
--------------------
N 2.533825191449017
E -0.4871945306631469
--------------------
---------

nodata vbw
nodata hdt
--------------------
nodata vbw
nodata hdt
--------------------
nodata vbw
nodata hdt
--------------------
nodata vbw
nodata hdt
--------------------
nodata hdt
nodata vtg
--------------------
nodata hdt
nodata vtg
--------------------
nodata hdt
nodata vtg
--------------------
nodata vtg
--------------------
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
-----------

nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw


nodata vbw
nodata vtg
--------------------
nodata vbw
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
nodata vtg
--------------------
nodata vbw
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
nodata hdt
--------------------
N

In [125]:
show=False
if show:
    plt.rcParams['font.family'] = 'MS Gothic' 
    fig, axes = plt.subplots(nrows=len(ggas), ncols=1, figsize=(8, 8*len(ggas)))

    for i in range(len(ggas)):
        ax = axes[i]
        ggas[i].plot.scatter(x='Lon', y='Lat',
                            marker='s', c='r', s=50, alpha=0.5, ax=ax, label='Deleted data')
        grid_cur_minute[i].plot.scatter(x='Lon', y='Lat',
                        marker='s', c='b', s=50, alpha=0.5, ax=ax, label='Used data')
        ax.set_title(path_name[i])
    plt.show()

In [126]:
def cur_minute_to_hour(grid_cur_m):
    grids0 = []
    grids1 = []
    curN = []
    curE = []
    timeHours = []
    UTC_time = []
    lats = []
    lons = []

    time_set = set(grid_cur_m["DtIdx"])
    for time in time_set:
        target = grid_cur_m[time == grid_cur_m["DtIdx"]]
        timeHours.append(time)
        UTC_time.append(target["UTC"].values[0])
        lats.append(np.mean(target["Lat"].values))
        lons.append(np.mean(target["Lon"].values))
        grids0.append(int(np.mean(target["Grid0"].values)))
        grids1.append(int(np.mean(target["Grid1"].values)))
        curN.append(np.mean(target["CurN"].values))
        curE.append( np.mean(target["CurE"].values))
#         lats.append(target.Lat.quantile(0.95))
#         lons.append(target.Lon.quantile(0.95))
#         grids0.append(int(target.Grid0.quantile(0.95)))
#         grids1.append(int(target.Grid1.quantile(0.95)))
#         curN.append(target.CurN.quantile(0.95))
#         curE.append(target.CurE.quantile(0.95))
    grid_cur = pd.DataFrame([])
    grid_cur["DtIdx"] = timeHours
    grid_cur["UTC"] = UTC_time
    grid_cur["CurN"] = curN
    grid_cur["CurE"] = curE
    grid_cur["Grid0"] = grids0
    grid_cur["Grid1"] = grids1
    grid_cur["Lat"] = lats
    grid_cur["Lon"] = lons
    return grid_cur

In [127]:
grid_curs = []
for grid_cur_m in grid_cur_minute:
    grid_cur = cur_minute_to_hour(grid_cur_m)
    grid_curs.append(grid_cur)

## データの保存

In [128]:
for i in range(len(grid_curs)):
    # 保存
    gird_cur_m = grid_cur_minute[i]
    grid_cur = grid_curs[i]
    grid_cur_m = grid_cur_m.sort_values('DtIdx_Minute')
    grid_cur_m = grid_cur_m.reset_index(drop=True)
    grid_cur = grid_cur.sort_values('DtIdx')
    grid_cur = grid_cur.reset_index(drop=True)

    day = path_name[i][-8:-6]
    save_path = osp.join(path_logs[i], f"cur_hours{dt_year}{dt_month:02}{day}.csv")
    grid_cur.to_csv(save_path)

    save_path = osp.join(path_logs[i], f"cur_minutes{dt_year}{dt_month:02}{day}.csv")
    grid_cur_m.to_csv(save_path)
    
    pkl.dump(curN_grid[i], open(osp.join(path_logs[i], f"curGridN.pkl"), 'wb'))
    pkl.dump(curE_grid[i], open(osp.join(path_logs[i], f"curGridE.pkl"), 'wb'))
    print(save_path)

E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_minutes201509.s.csv
E:\shunsukeE\data\shiplog/33東洋\2015\cur_

In [129]:
# 保存
# keys = grid_cur.keys()
# out = {}
# for k in keys:
#     out[k] = grid_cur[k]

# pkl.dump(out, open(osp.join(path_log, "cur.pkl"), 'wb'))

In [103]:
# pkl.dump(grid_cur, open(osp.join(path_log, "cur.pkl"), 'wb'))
# pkl.dump(grid_cur_m, open(osp.join(path_log, "cur_minus.pkl"), 'wb'))